# Parsing quartet likelihood-mapping results

This cleaned notebook retains only the Oweniidae and Sipuncula analyses reported in the manuscript. Other exploratory focal-clade analyses and all stored outputs have been removed.

Set the `ANALYSIS_ROOT` environment variable to the root of the original analysis workspace before adapting or running the example.

## Shared parser and metadata

In [ ]:
# Language: Python 3

import os
import re
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# These paths refer to the original analysis workspace, not the GitHub
# repository. Set ANALYSIS_ROOT before adapting this example.
ANALYSIS_ROOT = Path(
    os.environ.get("ANALYSIS_ROOT", "/path/to/original_analysis_workspace")
)
DATA_DIR = ANALYSIS_ROOT / "data"
SCRATCH_DIR = ANALYSIS_ROOT / "scratch"


def parse_quartet_table(input_file, dataset, mapping):
    """Parse IQ-TREE's quartet likelihood-mapping table."""
    columns = [
        "Index",
        "quartets",
        "a1_count",
        "a1_percent",
        "a2_count",
        "a2_percent",
        "a3_count",
        "a3_percent",
        "name",
    ]
    separator = (
        "-----------------------------------------------------------------------------"
    )
    pattern = re.compile(
        r"(\d+)\s+(\d+)\s+(\d+)\s+\((.*?)\s*\)\s+"
        r"(\d+)\s+\((.*?)\s*\)\s+(\d+)\s+\((.*?)\s*\)\s+(.*)"
    )

    rows = []
    in_table = False
    with open(input_file) as infile:
        for line in infile:
            line = line.strip()
            if separator in line:
                if in_table:
                    break
                in_table = True
                continue
            if in_table:
                match = pattern.match(line)
                if match:
                    rows.append(match.groups())

    table = pd.DataFrame(rows, columns=columns)
    table[columns[:-1]] = table[columns[:-1]].apply(
        pd.to_numeric, errors="coerce"
    )
    table.loc[table["quartets"] == 0, columns[2:-1]] = np.nan
    table["dataset"] = dataset
    table["mapping"] = mapping
    table = table.drop(columns="Index")

    palettes = {
        "a1_percent": sns.light_palette("seagreen", as_cmap=True),
        "a2_percent": sns.light_palette("darkblue", as_cmap=True),
        "a3_percent": sns.light_palette("darkred", as_cmap=True),
    }

    def get_color(percent, color_map):
        if pd.isna(percent) or percent < 100 / 3:
            return "#FFFFFF"
        scaled = (percent - 100 / 3) / (100 - 100 / 3)
        return mcolors.to_hex(color_map(scaled))

    support_columns = ["a1_percent", "a2_percent", "a3_percent"]
    table["dominant_area"] = table[support_columns].idxmax(
        axis=1, skipna=True
    )
    table["dominant_quartets"] = table[support_columns].max(
        axis=1, skipna=True
    )
    table["hexcolor"] = "#FFFFFF"
    for area in support_columns:
        mask = table["dominant_area"] == area
        table.loc[mask, "hexcolor"] = table.loc[mask, area].map(
            lambda value: get_color(value, palettes[area])
        )

    return table[
        [
            "name",
            "dominant_area",
            "dominant_quartets",
            "hexcolor",
            "dataset",
            "mapping",
        ]
    ]


def indexed_name_map(frame, index_column, value_column):
    """Build a mapping after normalizing numeric analysis identifiers."""
    identifiers = pd.to_numeric(
        frame[index_column], errors="coerce"
    ).astype("Int64").astype(str)
    mapping = dict(zip(identifiers, frame[value_column]))
    mapping.pop("<NA>", None)
    return mapping


metadata_dir = DATA_DIR / "gene_alignments"
meta = pd.read_csv(
    metadata_dir / "meta_species_map.txt",
    sep="\t",
    index_col=0,
)
L42meta = pd.read_csv(metadata_dir / "meta_L42.txt", sep="\t")
NPmeta = pd.read_csv(metadata_dir / "meta_NP.txt", sep="\t")

L42meta["ID"] = L42meta["ID"].astype(str)
NPmeta["ID"] = NPmeta["ID"].astype(str)
NPmeta["species"] = NPmeta["ID"]

species_by_matrix = {
    "SOGall": indexed_name_map(meta, "species_index", "species_name"),
    "SOG10_tax_occ": indexed_name_map(
        meta, "species_index", "species_name"
    ),
    "L42_synt": L42meta.set_index("ID")["species"].to_dict(),
    "L38_synt": L42meta.set_index("ID")["species"].to_dict(),
    "NP_synt": NPmeta.set_index("ID")["species"].to_dict(),
}

all_species = (
    list(meta["species_name"])
    + list(L42meta["species"])
    + list(NPmeta["species"])
)
all_clades = (
    list(meta["clade_name"])
    + list(L42meta["clade_name"])
    + list(NPmeta["clade_name"])
)
all_clean_names = (
    list(meta["species_name_clean"])
    + list(L42meta["species_name_clean"])
    + list(NPmeta["species_name_clean"])
)

clade_by_species = dict(zip(all_species, all_clades))
clean_name_by_species = dict(zip(all_species, all_clean_names))

## Collect Oweniidae and Sipuncula results

In [ ]:
# Language: Python 3

SUPERMATRICES = [
    "SOGall",
    "L42_synt",
    "L38_synt",
    "SOG10_tax_occ",
    "NP_synt",
]


def quartet_report_path(supermatrix, model, focal):
    """Return the IQ-TREE report path for one matrix/model/focal clade."""
    run_dir = SCRATCH_DIR / "tree_building" / supermatrix
    if supermatrix == "SOGall":
        run_dir = run_dir / "3plus_clades"
    filename = (
        f"{model}.quartet.quartet_map_{focal.lower()}."
        f"{supermatrix}_{focal}_quartets.iqtree"
    )
    return run_dir / filename


def collect_quartet_results(focal):
    """Collect the Oweniidae or Sipuncula mapping results across matrices."""
    frames = []
    for supermatrix in SUPERMATRICES:
        models = ["EDGEPROP"] if supermatrix == "SOGall" else [
            "SITEHET2",
            "EDGEPROP",
        ]
        for model in models:
            dataset = f"{supermatrix}.{model}"
            report = quartet_report_path(
                supermatrix, model, focal
            )
            parsed = parse_quartet_table(
                report, dataset=dataset, mapping=focal
            )
            parsed["species"] = (
                parsed["name"]
                .astype(str)
                .map(species_by_matrix[supermatrix])
            )
            frames.append(
                parsed[
                    [
                        "species",
                        "dominant_area",
                        "dominant_quartets",
                        "hexcolor",
                        "dataset",
                        "mapping",
                    ]
                ]
            )

    results = pd.concat(frames, ignore_index=True)
    results["clade"] = results["species"].map(clade_by_species)
    results = results.loc[
        results["dominant_quartets"].notna(),
        [
            "clade",
            "species",
            "dataset",
            "dominant_quartets",
            "hexcolor",
        ],
    ].copy()

    if focal == "Sip":
        results.loc[
            results["clade"].str.contains(
                "Echiura", case=False, na=False
            ),
            "clade",
        ] = "Thalassematidae"
        results.loc[
            results["clade"].str.contains(
                "Siboglinidae", case=False, na=False
            ),
            "clade",
        ] = "Siboglinidae"
        results.loc[
            results["clade"].isin(
                [
                    "Dinophilidae",
                    "Diurodrilidae",
                    "Orbiniidae",
                    "Spintheridae",
                    "Errantia:Myzostomida",
                    "Parergodrilidae",
                    "Nerillidae",
                ]
            ),
            "clade",
        ] = "Other Sedentaria"
        results.loc[
            results["clade"].str.contains(
                "Errantia", case=False, na=False
            ),
            "clade",
        ] = "Errantia"

    return results.sort_values(["clade", "species", "dataset"])


quartet_results = {
    focal: collect_quartet_results(focal)
    for focal in ["Owe", "Sip"]
}

## Plot the two focal analyses

In [ ]:
# Language: Python 3

CLADE_ORDER = {
    "Owe": [
        "Out:Sipuncula",
        "Out:Amphinomida",
        "Out:Chaetopteridae",
        "Out:Magelonidae",
        "Out:Oweniidae",
        "OutA:Nemertea",
        "OutA:Mollusca",
        "OutA:Brachiopoda",
        "OutA:Phoronida",
    ],
    "Sip": [
        "Capitellidae",
        "Opheliidae",
        "Thalassematidae",
        "Terebelliformia",
        "Arenicolidae",
        "Maldanidae",
        "Clitellata",
        "Cirratuliformia",
        "Siboglinidae",
        "Fabriciidae",
        "Sabellariidae",
        "Sabellidae",
        "Serpulidae",
        "Spionidae",
        "Other Sedentaria",
        "Errantia",
        "Out:Sipuncula",
        "Out:Amphinomida",
        "Out:Chaetopteridae",
        "Out:Magelonidae",
        "Out:Oweniidae",
    ],
}

FOCAL_LABEL = {
    "Owe": "Oweniidae",
    "Sip": "Sipuncula",
}


def plot_quartet_heatmap(
    results,
    focal,
    clade_order,
    output_path,
    include_unlisted_clades=True,
):
    """Plot the dominant quartet topology for every species and dataset."""
    data = results.copy()
    data["dominant_quartets"] = pd.to_numeric(
        data["dominant_quartets"], errors="coerce"
    )

    dataset_order = sorted(data["dataset"].unique())
    model_labels = {
        "EDGEPROP": "{MAT}+I+G4",
        "SITEHET2": "LG+C60+F+G4",
    }

    def dataset_label(dataset):
        supermatrix, model = dataset.split(".", 1)
        return f"{supermatrix};{model_labels.get(model, model)}"

    species_metadata = (
        data[["clade", "species"]]
        .drop_duplicates()
        .sort_values(["clade", "species"])
    )
    present_clades = species_metadata["clade"].drop_duplicates().tolist()
    ordered_clades = [
        clade for clade in clade_order if clade in present_clades
    ]
    if include_unlisted_clades:
        ordered_clades.extend(
            clade
            for clade in present_clades
            if clade not in ordered_clades
        )

    species_order = []
    for clade in ordered_clades:
        species_order.extend(
            species_metadata.loc[
                species_metadata["clade"] == clade,
                "species",
            ].tolist()
        )

    aggregated = (
        data.sort_values("dominant_quartets", ascending=False)
        .drop_duplicates(["species", "dataset"])
    )
    color_table = (
        aggregated.pivot(
            index="species",
            columns="dataset",
            values="hexcolor",
        )
        .reindex(index=species_order, columns=dataset_order)
        .fillna("#f0f0f0")
    )
    value_table = (
        aggregated.pivot(
            index="species",
            columns="dataset",
            values="dominant_quartets",
        )
        .reindex(index=species_order, columns=dataset_order)
    )

    n_rows, n_cols = color_table.shape
    rgba = np.zeros((n_rows, n_cols, 4), dtype=float)
    for row_index, species in enumerate(color_table.index):
        for column_index, dataset in enumerate(color_table.columns):
            rgba[row_index, column_index, :] = mcolors.to_rgba(
                color_table.loc[species, dataset]
            )

    cell_width, cell_height = 0.7, 0.4
    figure_width = max(6, n_cols * cell_width + 4.5)
    figure_height = max(4, n_rows * cell_height)
    figure = plt.figure(figsize=(figure_width, figure_height))
    grid = figure.add_gridspec(
        nrows=1,
        ncols=2,
        width_ratios=[1.0, 0.6],
        wspace=0.7,
    )

    axis = figure.add_subplot(grid[0, 0])
    axis.imshow(
        rgba,
        aspect="auto",
        interpolation="none",
        origin="upper",
    )
    axis.set_xticks(np.arange(n_cols))
    axis.set_xticklabels(
        [dataset_label(dataset) for dataset in dataset_order],
        rotation=45,
        ha="right",
        color="black",
    )
    axis.set_yticks(np.arange(n_rows))
    ytick_labels = axis.set_yticklabels(
        [
            clean_name_by_species.get(species, species)
            for species in color_table.index
        ],
        color="black",
    )
    for label in ytick_labels:
        label.set_fontstyle("italic")

    axis.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    axis.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    axis.grid(which="minor", color="white", linewidth=0.5)
    axis.tick_params(which="minor", bottom=False, left=False)

    species_counts = (
        species_metadata.groupby("clade")["species"].nunique().to_dict()
    )
    clade_counts = [
        species_counts[clade]
        for clade in ordered_clades
        if clade in species_counts
    ]
    boundaries = np.cumsum(clade_counts) - 0.5
    for boundary in boundaries[:-1]:
        axis.hlines(
            boundary,
            -0.5,
            n_cols - 0.5,
            colors="black",
            linewidth=1,
        )

    clade_centers = (
        np.cumsum(clade_counts)
        - np.array(clade_counts) / 2
        - 0.5
    )
    for center, clade in zip(clade_centers, ordered_clades):
        displayed_clade = (
            clade.split(":", 1)[1] if ":" in clade else clade
        )
        axis.text(
            1.01,
            center,
            displayed_clade,
            transform=axis.get_yaxis_transform(),
            ha="left",
            va="center",
            fontsize=10,
            color="black",
            clip_on=False,
        )

    for row_index in range(n_rows):
        for column_index in range(n_cols):
            value = value_table.iat[row_index, column_index]
            if pd.notna(value):
                axis.text(
                    column_index,
                    row_index,
                    f"{value:.0f}",
                    ha="center",
                    va="center",
                    color="black",
                    fontsize=9,
                )

    axis.set_xlim(-0.5, n_cols - 0.5)
    axis.set_ylim(n_rows - 0.5, -0.5)
    axis.set_xlabel("Dataset")
    axis.set_ylabel("Species")
    axis.set_title(
        f"Quartet likelihood mapping for {FOCAL_LABEL[focal]}"
    )

    legend_axis = figure.add_subplot(grid[0, 1])
    cm_in_inches = 1 / 2.54
    desired_height = 3 * cm_in_inches
    position = legend_axis.get_position()
    height_fraction = min(
        0.95, desired_height / figure.get_size_inches()[1]
    )
    center_y = position.y0 + position.height / 2
    legend_axis.set_position(
        [
            position.x0,
            max(0.02, center_y - height_fraction / 2),
            position.width,
            height_fraction,
        ]
    )

    palettes = {
        "a1_percent": sns.light_palette("seagreen", as_cmap=True),
        "a2_percent": sns.light_palette("darkblue", as_cmap=True),
        "a3_percent": sns.light_palette("darkred", as_cmap=True),
    }
    gradient_positions = np.linspace(32, 100, 10)
    scaled_positions = np.clip(
        (gradient_positions - 33) / (100 - 33),
        0,
        1,
    )
    legend_rows = ["a1_percent", "a2_percent", "a3_percent"]
    bars = np.zeros((len(legend_rows), 10, 4), dtype=float)
    for row_index, support_column in enumerate(legend_rows):
        bars[row_index, :, :] = palettes[support_column](
            scaled_positions
        )

    legend_axis.imshow(
        bars,
        aspect="auto",
        interpolation="nearest",
        origin="upper",
        extent=[0, 100, 0, 3],
    )
    legend_axis.set_yticks([0.5, 1.5, 2.5])
    legend_axis.set_yticklabels(["A3", "A2", "A1"])
    legend_ticks = [0, 25, 50, 75, 100]
    legend_axis.set_xticks(legend_ticks)
    legend_axis.set_xticklabels(
        [
            f"{(100 / 3) + tick * (2 / 3):.0f}%"
            for tick in legend_ticks
        ]
    )
    legend_axis.set_xlabel("Quartet fraction")
    legend_axis.set_title("Dominant topology", pad=8)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(
        output_path,
        format="svg",
        bbox_inches="tight",
    )
    plt.close(figure)


for focal, results in quartet_results.items():
    plot_quartet_heatmap(
        results=results,
        focal=focal,
        clade_order=CLADE_ORDER[focal],
        output_path=(
            DATA_DIR
            / "trees"
            / f"{focal}_heatmap_dominant_quartets_v2.svg"
        ),
    )